# Kirinyaga Tomato Market Data  Dataset Exploration
### Semester Project: Reducing Broker Exploitation of Tomato Farmers in Kirinyaga County (ML + Mobile App)

This notebook covers the assignment deliverable: identifying open-source datasets relevant to the
research problem, naming a candidate output variable for each, selecting the most appropriate
dataset and carrying out initial data exploration.


## Part 1 & 2  Candidate Open-Source Datasets and Candidate Output Variables

|  | Dataset | Source | Candidate output variable | Notes |
|---|---|---|---|---|
| 1 | **KAMIS Kirinyaga Tomato Market Prices** | Kenya Agricultural Market Information System (kamis.kilimo.go.ke) | `Retail_Wholesale_Spread_KSh` / `Spread_Pct_of_Wholesale` (broker-margin proxy) | Real wholesale + retail prices + supply volume, Kirinyaga-specific, ~2 years daily data. **Selected dataset  see below.** |
| 2 | WFP Kenya Food Prices | World Food Programme via HDX (data.humdata.org) | Commodity price (KSh/kg) | Long national time series (2006–2024); needs verification that tomatoes/Kirinyaga-area markets are tracked. |
| 3 | KNBS County Statistical Abstract  Kirinyaga | Kenya National Bureau of Statistics (knbs.or.ke) | Production volume (Tonnes) or Area Cropped (Ha) | Useful as an input feature (supply context) rather than a direct output variable. |
| 4 | KNBS Consumer Price Index (CPI) Tomatoes | KNBS (knbs.or.ke) | National tomato retail price index | Macro-level cross-check series, not county-specific. |
| 5 | Open-Meteo / NASA POWER Historical Weather | open-meteo.com / power.larc.nasa.gov | Not an output variable  a supporting **input feature** (rainfall, temperature) that helps explain price shocks unrelated to broker behaviour. |
| 6 | Primary Farmer Survey (to be collected) | Own data collection, Kirinyaga farmers | Farm-gate price paid to farmer (KSh/kg) | The only source that can directly measure what a farmer actually received  needed to validate the "exploitation gap" label. |
| # | Dataset | Source | Notes |
| 1 | **KAMIS Kirinyaga Tomato Market Prices**  Kenya Agricultural Market Information System (kamis.kilimo.go.ke)  `Retail_Wholesale_Spread_KSh` / `Spread_Pct_of_Wholesale` (broker-margin proxy)  Real wholesale + retail prices + supply volume, Kirinyaga-specific, ~2 years daily data. **Selected dataset  see below.** |
| 2 | WFP Kenya Food Prices  World Food Programme via HDX (data.humdata.org)  Commodity price (KSh/kg) Long national time series (2006–2024); needs verification that tomatoes/Kirinyaga-area markets are tracked. |
| 3 | KNBS County Statistical Abstract  Kirinyaga  Kenya National Bureau of Statistics (knbs.or.ke)  Production volume (Tonnes) or Area Cropped (Ha)  Useful as an input feature (supply context) rather than a direct output variable. |
| 4 | KNBS Consumer Price Index (CPI) Tomatoes  KNBS (knbs.or.ke)  National tomato retail price index  Macro-level cross-check series, not county-specific. |
| 5 | Open-Meteo / NASA POWER Historical Weather  open-meteo.com / power.larc.nasa.gov  Not an output variable  a supporting **input feature** (rainfall, temperature) that helps explain price shocks unrelated to broker behaviour. |
| 6 | Primary Farmer Survey (to be collected)  Own data collection, Kirinyaga farmers  Farm-gate price paid to farmer (KSh/kg)  The only source that can directly measure what a farmer actually received  needed to validate the "exploitation gap" label. |

**Why most datasets don't have a clean "output variable":** several of the above (weather,
production volume, CPI) are better used as *input features* that help the model explain price
movement, rather than as the target the model predicts. Only the KAMIS price data and the
(not-yet-collected) farmer survey directly measure something that could serve as a genuine
prediction target for a broker-exploitation model.


## Part 3  Selected Dataset

**Selected: KAMIS Kirinyaga Tomato Market Prices (cleaned).**

Reasons: it is real (not simulated), Kirinyaga-specific, covers 2 years of near-daily
observations across 5 markets (Kagio, Kerugoya, Kutus, Makutano Kirinyaga, Ngurubani Market)
and already includes both `Wholesale` and `Retail` prices  letting us compute a
**broker-margin proxy** (`Retail_Wholesale_Spread_KSh` and `Spread_Pct_of_Wholesale`), which is
the closest available signal to the exploitation gap this project investigates.

The file was already cleaned in a prior step: duplicate rows removed, a data-entry artifact
(`County == "test"`) dropped, price strings (e.g. `"15.00/Kg"`) parsed to numeric KSh/kg, and the
spread columns derived.


## Load the Dataset



In [ ]:
import pandas as pd

# Colab file upload (skip this try block if running locally with the file already present)
try:
    from google.colab import files
    uploaded = files.upload()
    file_path = list(uploaded.keys())[0]
except ImportError:
    file_path = "Kirinyaga Tomato Prices.xlsx"

df = pd.read_excel(file_path, sheet_name="Kirinyaga", header=3)
df.head()


,Date,County,Market,Wholesale KSh per Kg,Retail KSh per Kg,Retail Wholesale Spread KSh,Spread Pct of Wholesale,Supply Volume Kg
0,2025-05-16,Kirinyaga,Kagio,44.12,58.82,14.70,0.333182,28000.0
1,2025-05-20,Kirinyaga,Kagio,50.00,64.71,14.71,0.294200,28000.0
2,2025-07-04,Kirinyaga,Kagio,30.00,60.00,30.00,1.000000,50000.0
3,2025-07-11,Kirinyaga,Kagio,50.00,80.00,30.00,0.600000,50000.0
4,2025-07-19,Kirinyaga,Kagio,50.00,80.00,30.00,0.600000,10000.0


### a. How many rows and columns are contained in the dataset? (3 marks)

In [2]:
n_rows, n_cols = df.shape
print(f"The dataset contains {n_rows} rows and {n_cols} columns.")
df.shape


The dataset contains 478 rows and 8 columns.


(478, 8)

### b. What datatypes are contained in the dataset? (3 marks)

In [3]:
print(df.dtypes)


Date                           datetime64[us]
County                                    str
Market                                    str
Wholesale KSh per Kg                  float64
Retail KSh per Kg                     float64
Retail Wholesale Spread KSh           float64
Spread Pct of Wholesale               float64
Supply Volume Kg                      float64
dtype: object


### c. Is the dataset complete? i.e. no missing values? (2 marks)

In [ ]:
missing = df.isna().sum()
print("Missing values per column:")
print(missing)
print()
total_missing = missing.sum()
if total_missing == 0:
    print("The dataset is complete  no missing values.")
else:
    print(f"The dataset is NOT complete  {total_missing} missing value(s) found, "
          f"in column(s): {list(missing[missing > 0].index)}")


Missing values per column:
Date                            0
County                          0
Market                          0
Wholesale KSh per Kg            0
Retail KSh per Kg               0
Retail Wholesale Spread KSh     0
Spread Pct of Wholesale         0
Supply Volume Kg               13
dtype: int64

The dataset is NOT complete — 13 missing value(s) found, in column(s): ['Supply Volume Kg']


### d. Slice out the first 15 rows and last 20 rows and merge them into a new dataframe titled "df_sample". Display this new dataframe. (4 marks)

In [5]:
df_sample = pd.concat([df.head(15), df.tail(20)])
df_sample


,Date,County,Market,Wholesale KSh per Kg,Retail KSh per Kg,Retail Wholesale Spread KSh,Spread Pct of Wholesale,Supply Volume Kg
0,2025-05-16,Kirinyaga,Kagio,44.12,58.82,14.70,0.333182,28000.0
1,2025-05-20,Kirinyaga,Kagio,50.00,64.71,14.71,0.294200,28000.0
2,2025-07-04,Kirinyaga,Kagio,30.00,60.00,30.00,1.000000,50000.0
3,2025-07-11,Kirinyaga,Kagio,50.00,80.00,30.00,0.600000,50000.0
4,2025-07-19,Kirinyaga,Kagio,50.00,80.00,30.00,0.600000,10000.0
5,2025-07-25,Kirinyaga,Kagio,40.00,60.00,20.00,0.500000,10000.0
6,2025-08-15,Kirinyaga,Kagio,50.00,80.00,30.00,0.600000,10000.0
7,2025-08-22,Kirinyaga,Kagio,30.00,50.00,20.00,0.666667,50000.0
8,2025-08-29,Kirinyaga,Kagio,50.00,120.00,70.00,1.400000,35000.0
9,2025-09-06,Kirinyaga,Kagio,42.86,120.00,77.14,1.799813,20000.0
